IMPORTING THE LIBRARIES

In [ ]:
import numpy as np                # Array operations; most sklearn outputs are numpy arrays
import matplotlib.pyplot as plt   # Not used here, but standard to import for plotting
import pandas as pd               # Reading CSV files and DataFrame manipulation

IMPORTING THE DATASET

In [15]:
dataset = pd.read_csv('Data.csv')

# iloc = integer location; selects by row/column position (not label)
# [:, :-1] → all rows, all columns EXCEPT the last one → feature matrix X
# [:, -1]  → all rows, last column only              → target vector y
# .values converts the DataFrame to a raw NumPy array
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [16]:
# X has 3 columns: Country (str), Age (float), Salary (float)
# Notice two NaN values: one in Age (row 6), one in Salary (row 4)
print(X)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 nan]
 ['France' 35.0 58000.0]
 ['Spain' nan 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]


In [17]:
# y is the target: 'Yes'/'No' strings — will be label-encoded to 1/0 later
print(y)

<StringArray>
['No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes']
Length: 10, dtype: str


HANDLING MISSING DATA

In [18]:
from sklearn.impute import SimpleImputer

# Replace NaN values with the column mean
# Other strategies: 'median', 'most_frequent', 'constant'
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

# IMPORTANT: fit() learns the mean from the data (does NOT change X yet)
# We only apply it to numerical columns [1:3] = Age and Salary
# Column 0 (Country) is a string — can't compute a mean on it
# Note: [1:3] is exclusive of index 3, so it selects columns 1 and 2 only
imputer.fit(X[:, 1:3])

# transform() applies the learned means and returns the filled array
# We write the result back into X to update it in-place
X[:, 1:3] = imputer.transform(X[:, 1:3])

In [19]:
# NaNs are now replaced:
# Row 4, Salary: was NaN → now 63777.78 (mean of all salary values)
# Row 6, Age:    was NaN → now 38.78    (mean of all age values)
print(X)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 63777.77777777778]
 ['France' 35.0 58000.0]
 ['Spain' 38.77777777777778 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]


ENCODING CATEGORICAL DATA

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# OneHotEncoder: converts a categorical column into N binary columns (one per category)
# e.g. 'France' → [1,0,0], 'Germany' → [0,1,0], 'Spain' → [0,0,1]
# This avoids implying any false numeric ordering (France=0 < Germany=1 would be wrong)
#
# ColumnTransformer applies different transformations to different columns:
#   - ('encoder', OneHotEncoder(), [0]) → apply OHE to column 0 (Country)
#   - remainder='passthrough'           → leave all other columns (Age, Salary) unchanged
#   - remainder='drop' would DELETE the other columns instead
#
# [0] is a list of column indices to transform; you can pass multiple e.g. [0, 2]
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')

# fit_transform() = fit() + transform() in one call
# Wrap in np.array() because ColumnTransformer returns a sparse matrix by default
X = np.array(ct.fit_transform(X))

In [21]:
# X now has 5 columns instead of 3:
# [France, Germany, Spain, Age, Salary]
# The original Country column is gone, replaced by 3 binary columns
print(X)

[[1.0 0.0 0.0 44.0 72000.0]
 [0.0 0.0 1.0 27.0 48000.0]
 [0.0 1.0 0.0 30.0 54000.0]
 [0.0 0.0 1.0 38.0 61000.0]
 [0.0 1.0 0.0 40.0 63777.77777777778]
 [1.0 0.0 0.0 35.0 58000.0]
 [0.0 0.0 1.0 38.77777777777778 52000.0]
 [1.0 0.0 0.0 48.0 79000.0]
 [0.0 1.0 0.0 50.0 83000.0]
 [1.0 0.0 0.0 37.0 67000.0]]


In [22]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder: maps string classes to integers alphabetically
# 'No' → 0, 'Yes' → 1
#
# WHY LabelEncoder for y but NOT for X?
# LabelEncoder is fine for a binary target because there's no ordering issue.
# On feature columns with 3+ categories (like Country), it would wrongly imply
# France(0) < Germany(1) < Spain(2) — that's why we used OneHotEncoder for X.
le = LabelEncoder()
y = le.fit_transform(y)

In [23]:
# y is now a numeric array: 0 = No, 1 = Yes
print(y)

[0 1 0 0 1 1 0 1 0 1]


SPLITTING DATASET INTO TRAINING AND TEST SET

In [24]:
from sklearn.model_selection import train_test_split

# Split BEFORE feature scaling — critical rule!
# If you scale first, the scaler sees test data during fit() → data leakage.
# The test set must simulate truly unseen data at all times.
#
# test_size=0.2  → 20% test (2 rows), 80% train (8 rows)
# random_state=1 → fixes the random seed so the split is reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [25]:
print(X_train)  # 8 rows, 5 columns

[[0.0 0.0 1.0 38.77777777777778 52000.0]
 [0.0 1.0 0.0 40.0 63777.77777777778]
 [1.0 0.0 0.0 44.0 72000.0]
 [0.0 0.0 1.0 38.0 61000.0]
 [0.0 0.0 1.0 27.0 48000.0]
 [1.0 0.0 0.0 48.0 79000.0]
 [0.0 1.0 0.0 50.0 83000.0]
 [1.0 0.0 0.0 35.0 58000.0]]


In [26]:
print(X_test)   # 2 rows, 5 columns

[[0.0 1.0 0.0 30.0 54000.0]
 [1.0 0.0 0.0 37.0 67000.0]]


In [27]:
print(y_train)

[0 1 0 0 1 1 0 1]


In [28]:
print(y_test)

[0 1]


FEATURE SCALING

In [29]:
from sklearn.preprocessing import StandardScaler

# StandardScaler: transforms each value to z = (x - mean) / std
# Result: each column has mean=0, std=1
# WHY? Prevents high-magnitude features (Salary ~60k) from dominating
# low-magnitude ones (Age ~30-50) in distance-based models (KNN, SVM, etc.)
#
# THE GOLDEN RULE:
#   fit_transform() on X_train → learns mean & std FROM TRAINING DATA ONLY
#   transform()     on X_test  → applies the SAME mean & std (no re-learning)
# Never fit on X_test! That would leak test statistics into preprocessing.
#
# Why [:, 3:] and not all columns?
# Columns 0,1,2 are one-hot encoded (already 0 or 1) — scaling them is
# unnecessary and makes them harder to interpret. Only scale Age and Salary.
sc = StandardScaler()
X_train[:, 3:] = sc.fit_transform(X_train[:, 3:])
X_test[:, 3:]  = sc.transform(X_test[:, 3:])

In [30]:
# Age and Salary are now standardized; one-hot columns (0/1) are unchanged
print(X_train)

[[0.0 0.0 1.0 -0.19159184384578545 -1.0781259408412425]
 [0.0 1.0 0.0 -0.014117293757057777 -0.07013167641635372]
 [1.0 0.0 0.0 0.566708506533324 0.633562432710455]
 [0.0 0.0 1.0 -0.30453019390224867 -0.30786617274297867]
 [0.0 0.0 1.0 -1.9018011447007988 -1.420463615551582]
 [1.0 0.0 0.0 1.1475343068237058 1.232653363453549]
 [0.0 1.0 0.0 1.4379472069688968 1.5749910381638885]
 [1.0 0.0 0.0 -0.7401495441200351 -0.5646194287757332]]


In [31]:
# X_test scaled using the SAME scaler fitted on X_train (not re-fitted)
print(X_test)

[[0.0 1.0 0.0 -1.4661817944830124 -0.9069571034860727]
 [1.0 0.0 0.0 -0.44973664397484414 0.2056403393225306]]
